In [1]:
import psycopg2
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv('/Users/janhavihatekar/Desktop/threat-detection/backend/.env')

conn = psycopg2.connect(os.getenv("DATABASE_URL"))

df = pd.read_sql("SELECT * FROM logs", conn)
print(f"Total logs: {len(df)}")
print(df.head())


Total logs: 9782
   id            ip    endpoint method  status_code  response_time  \
0   1  192.168.1.10   /api/data    GET          200            259   
1   2  192.168.1.10     /logout    GET          200            128   
2   3  192.168.1.10   /api/data    GET          200            232   
3   4  192.168.1.11  /api/users    GET          200            220   
4   5      10.0.0.5  /dashboard    GET          200            264   

                        created_at  
0 2026-05-16 04:27:33.283238+00:00  
1 2026-05-16 04:27:33.292456+00:00  
2 2026-05-16 04:27:33.293110+00:00  
3 2026-05-16 04:27:33.293720+00:00  
4 2026-05-16 04:27:33.294277+00:00  


/var/folders/qk/cnl9x8sx5snbqsvbvg7fvdsr0000gn/T/ipykernel_13553/2756858778.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM logs", conn)


In [2]:
# Har IP ka behavior aggregate karo
features = df.groupby('ip').agg(
    total_requests=('id', 'count'),
    failed_requests=('status_code', lambda x: (x == 401).sum()),
    unique_endpoints=('endpoint', 'nunique'),
    avg_response_time=('response_time', 'mean')
).reset_index()

# Failed ratio calculate karo
features['failed_ratio'] = features['failed_requests'] / features['total_requests']

print(features)


               ip  total_requests  failed_requests  unique_endpoints  \
0        10.0.0.5            2233                0                 5   
1        10.0.0.8            2287                0                 5   
2   185.220.101.5             171              171                 1   
3    192.168.1.10            2411                0                 5   
4    192.168.1.11            2321                0                 5   
5    203.45.67.89             162              162                 1   
6  45.142.212.100             197              197                 1   

   avg_response_time  failed_ratio  
0         175.395432           0.0  
1         176.132925           0.0  
2           9.777778           1.0  
3         180.907922           0.0  
4         172.932788           0.0  
5           9.401235           1.0  
6           9.512690           1.0  


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import numpy as np

X = features[['total_requests', 'failed_ratio', 'unique_endpoints', 'avg_response_time']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    contamination=0.05,
    random_state=42
)
model.fit(X_scaled)

features['prediction'] = model.predict(X_scaled)
features['anomaly_score'] = model.score_samples(X_scaled)
features['threat_score'] = ((features['anomaly_score'] * -1) * 50 + 50).clip(0, 100)

print("🔍 Results:")
print(features[['ip', 'failed_ratio', 'threat_score', 'prediction']].sort_values('threat_score', ascending=False))

🔍 Results:
               ip  failed_ratio  threat_score  prediction
6  45.142.212.100           1.0     77.486765          -1
3    192.168.1.10           0.0     77.111288           1
2   185.220.101.5           1.0     76.925477           1
5    203.45.67.89           1.0     76.618618           1
4    192.168.1.11           0.0     75.251499           1
0        10.0.0.5           0.0     73.039018           1
1        10.0.0.8           0.0     71.020390           1
